# Reliability-weighted geometry: component validation

This notebook reruns the frozen **192-case synthetic component experiment**, not model training. CPU is sufficient. It includes heldout generator seeds and stable-but-wrong counterexamples. Weighted label error must not be described as improved model accuracy.

Upload this notebook to Colab using **File > Upload notebook**, then run the cells and select the accompanying `reliability_v8_colab_bundle.zip` when requested. The bundle contains code, protocol and scene metadata; no raw RGB-D images, checkpoints or credentials. Local source/bundle checks were performed; this notebook has not been executed on a Google-managed runtime.

The real 128-scene audit and GPU integration check require the separate local V7 assets and are not run here. Colab resources are variable; see the [official FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
import sys, subprocess, importlib.util, importlib.metadata, platform
for module, package in [('numpy','numpy'),('scipy','scipy'),('matplotlib','matplotlib'),('torch','torch')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', package], check=True)
print('Python', platform.python_version())
for package in ['numpy','scipy','matplotlib','torch']:
    print(package, importlib.metadata.version(package))


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, hashlib, os, json
uploaded = files.upload()
expected = 'reliability_v8_colab_bundle.zip'
if expected not in uploaded:
    raise ValueError('Select the accompanying ' + expected)
payload = uploaded[expected]
assert hashlib.sha256(payload).hexdigest() == '4654c3bbc5f11e3340a6d9a41ae83fbbcd09c0d8e306fa5b1a077b95e835f35d', 'Bundle differs from delivered source'
project = Path('/content/elec4240_reliability/workspace/repo')
project.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for member in archive.infolist():
        target = (project / member.filename).resolve()
        assert target.is_relative_to(project.resolve()), 'Unsafe archive path'
    archive.extractall(project)
os.chdir(project)
sys.path.insert(0, str(project))
from reliability_v8.study import verify_protocol
print('Frozen source verified:', verify_protocol()['version'])


In [ ]:
subprocess.run([sys.executable, '-m', 'reliability_v8.test_core'], check=True)
subprocess.run([sys.executable, '-m', 'reliability_v8.study', 'synthetic'], check=True)


In [ ]:
result_dir = project / 'results/reliability_v8'
data = json.loads((result_dir / 'synthetic.json').read_text())
held = [r for r in data['records'] if r['split'] == 'heldout']
print('Cases:', len(data['records']), '| Heldout-generator cases:', len(held))
for condition in ['clean','iid_noise','heterogeneous_noise','outliers','holes','smooth_bias']:
    rows = [r['metrics']['all'] for r in held if r['corruption'] == condition]
    means = {k: sum(r[k] for r in rows)/len(rows) for k in
             ['uniform_error_deg','weighted_error_deg','shuffled_error_deg']}
    print(condition, means)
print('These are supervision-selection errors, not trained-model accuracy. No real-data gate is evaluated here.')
from IPython.display import display, Image
display(Image(filename=str(result_dir / 'figures/synthetic_examples.png')))
environment = {'python':platform.python_version(),
               'packages':{p:importlib.metadata.version(p) for p in ['numpy','scipy','matplotlib','torch']}}
(result_dir / 'colab_environment.json').write_text(json.dumps(environment, indent=2))


In [ ]:
archive_path = Path('/content/reliability_v8_colab_results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in result_dir.rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(result_dir))
files.download(str(archive_path))
